In [9]:
from sympy.physics.units import temperature
!pip install "pydantic>=2.11,<3" "pydantic-settings>=2.5,<3"
!pip install -U langchain langchain-core langchain-community langchain-ollama langchain-mcp-adapters mcp
!pip install typing-extensions

In [2]:
import sys, pydantic, pydantic_settings
print("python:", sys.executable)


python: C:\Users\alombard.PCP-GI-05\miniconda3\python.exe


# 0 - Quick start

In [3]:
import requests

In [21]:
response = requests.post("http://localhost:11434/api/generate", data='{"model": "gemma3:4b", "prompt": "Why is the sky blue?"}')

In [30]:
print(response.content)

b'{"model":"gemma3:4b","created_at":"2025-10-22T06:37:01.4341258Z","response":"The","done":false}\n{"model":"gemma3:4b","created_at":"2025-10-22T06:37:01.4783097Z","response":" blue","done":false}\n{"model":"gemma3:4b","created_at":"2025-10-22T06:37:01.5211947Z","response":" color","done":false}\n{"model":"gemma3:4b","created_at":"2025-10-22T06:37:01.5665735Z","response":" of","done":false}\n{"model":"gemma3:4b","created_at":"2025-10-22T06:37:01.6083497Z","response":" the","done":false}\n{"model":"gemma3:4b","created_at":"2025-10-22T06:37:01.6509267Z","response":" sky","done":false}\n{"model":"gemma3:4b","created_at":"2025-10-22T06:37:01.6966599Z","response":" is","done":false}\n{"model":"gemma3:4b","created_at":"2025-10-22T06:37:01.7410093Z","response":" a","done":false}\n{"model":"gemma3:4b","created_at":"2025-10-22T06:37:01.7903125Z","response":" fascinating","done":false}\n{"model":"gemma3:4b","created_at":"2025-10-22T06:37:01.8420777Z","response":" phenomenon","done":false}\n{"mod

# 1 - Using LangChain

In [31]:
from langchain_community.llms import Ollama

In [34]:
model = Ollama(model="gemma3:4b", top_k=5, system="You are a funny kid.", temperature=1.2)

In [35]:
model.invoke("Why is the sky blue?")

"Okay, okay, this is a *super* important question! \n\nIt’s because the sun is a giant, sparkly, yellow ball of awesome! But when it shines through the air, it bumps into tiny bits of air and... guess what? Blue light is like a bouncy ball – it bounces *everywhere*! \n\nThe other colors, like red and orange, are like grumpy dudes – they don't bounce as much. So we see blue bouncing all over the place, and that's why the sky is blue! \n\nIt’s basically a giant, bouncy blue party! 🎉 \n\nDoes that make sense? Or do you want me to explain it with even *more* bouncy balls? 😄"

# Agentic AI and MCP

In [36]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_mcp_adapters.client import MultiServerMCPClient

In [18]:
model = ChatOllama(model="qwen3:4b", reasoning=True)

mcp_client = MultiServerMCPClient({
    "math": {
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8000/mcp"
    }
})

tools = await mcp_client.get_tools()

model.bind_tools(tools)

agent = create_agent(
    model,
    tools,
    system_prompt="You are an helpful assistant. You use MCP tools when available.")

result = await agent.ainvoke({
    "messages": [
        {"role": "user", "content": "Can you compute 1205235 + 944345 ?"}]})

print(result)

{'messages': [HumanMessage(content='Can you compute 1205235 + 944345 ?', additional_kwargs={}, response_metadata={}, id='15cee6c8-da1e-49d5-9931-676b0e001981'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking to compute 1205235 plus 944345. Let me check the tools provided.\n\nLooking at the tools section, there\'s a function called add that takes two integers a and b. The description says it makes the sum of two numbers. Perfect, this is exactly what the user needs.\n\nSo I need to call the add function with a=1205235 and b=944345. Let me verify the parameters: both are integers, which they are here. The function requires a and b as integers, so this should work.\n\nI don\'t see any other tools related to weather or other functions that would be relevant here. The user is just asking for a simple addition. So the correct tool to use is the add function with those two numbers.\n\nI\'ll structure the tool call as a JSON object inside the tool_call 

In [20]:
result = await agent.ainvoke({"messages": [
        {"role": "user", "content": "What is the weather in Belfort?"}]})

print(result)

{'messages': [HumanMessage(content='What is the weather in Belfort?', additional_kwargs={}, response_metadata={}, id='89978e6e-224a-4a98-a653-0261f0b5959b'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user is asking for the weather in Belfort. Wait, first I need to check if that\'s the correct spelling. Belfort? I think the correct city name might be Besançon or maybe it\'s a different spelling. Wait, no, Belfort is a city in France. Let me confirm. Oh, but the user wrote "Belfort" with a \'t\' at the end. Hmm, maybe it\'s a typo for "Belfort" but in English, it\'s often spelled "Belfort" (like the city in France). Wait, actually, the city in France is called Belfort, so maybe the user meant that.\n\nBut regardless, the tool provided is the \'weather\' function which takes a location as a string. So the user is asking for the weather in Belf: wait, no, the user said "Belfort". Let me check the spelling again. The function requires a location st